# Setup Instructions

```bash
mkvirtualenv skan-dev
git clone git@github.com:AFM-SPM/TopoStats.git && cd TopoStats && pip install -e .[dev,tests,notebooks]
git clone git@github.com:ns-rse/skan.git && cd skan && pip install -e .[all]
git clone git@github.com:napari/napari.git && cd napari && pip install .
pip install pyqt5
```

In [ ]:
%gui qt
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from skimage.measure import label
from skimage import feature, filters, morphology
import skimage

import skan
from skan import draw
from skan.csr import skeleton_to_csgraph
from skan import Skeleton, summarize

from topostats.filters import Filters
from topostats.grains import Grains
from topostats.grainstats import GrainStats
from topostats.io import find_files, read_yaml, write_yaml, LoadScans
from topostats.logs.logs import setup_logger, LOGGER_NAME
from topostats.plottingfuncs import Images
from topostats.tracing.dnatracing import dnaTrace, prep_arrays
from topostats.utils import update_config

CURRENT_DIR = Path().cwd()
BASE_DIR = CURRENT_DIR.parents[0]
RESOURCES = BASE_DIR / "tests" / "resources"
FILE_EXT = ".spm"
image_files = find_files(base_dir=BASE_DIR / "tests", file_ext=FILE_EXT)
config = read_yaml(BASE_DIR / "topostats" / "default_config.yaml")
config["log_level"] = "quiet"
loading_config = config["loading"]
filter_config = config["filter"]
filter_config.pop("run")
grain_config = config["grains"]
grain_config.pop("run")
grainstats_config = config["grainstats"]
grainstats_config.pop("run")
dnatracing_config = config["dnatracing"]
dnatracing_config.pop("run")
plotting_config = config["plotting"]
plotting_config.pop("run")
plotting_config["image_set"] = "all"



In [ ]:
# Loads scans
all_scan_data = LoadScans(image_files, **config["loading"])
all_scan_data.get_data()
# Filter image
filtered_image = Filters(
    image=all_scan_data.img_dict["minicircle"]["image_original"],
    filename=all_scan_data.img_dict["minicircle"]["img_path"],
    pixel_to_nm_scaling=all_scan_data.img_dict["minicircle"]["pixel_to_nm_scaling"],
    **filter_config,
)
filtered_image.filter_image()
# Find Grains
grains = Grains(
    image=filtered_image.images["zero_average_background"],
    filename=filtered_image.filename,
    pixel_to_nm_scaling=filtered_image.pixel_to_nm_scaling,
    **grain_config,
)
grains.find_grains()

# Plot some images
fig, ax = Images(
    data=grains.directions["above"]["coloured_regions"],
    filename="minicircle_coloured_regions",
    output_dir="img/",
    save=True,
    **plotting_config,
).plot_and_save()
binary_mask = np.where(grains.directions["above"]["coloured_regions"] == 0, 0, 1)
fig, ax = Images(
    data=binary_mask,
    filename="minicircle_binary_mask",
    output_dir="img/",
    save=True,
    **plotting_config,
).plot_and_save()

# Crop arrays so that we have smaller objects to work with
cropped_images, cropped_masks = prep_arrays(
    image=filtered_image.images["zero_average_background"],
    labelled_grains_mask=grains.directions["above"]["labelled_regions_02"],
    pad_width=2,
)
plt.imshow(cropped_images[3])
# Skeletonize the grains
skeletons = [morphology.skeletonize(grain) for grain in cropped_masks]

### SKAN
SKELETONIZE_METHOD = "zhang"

In [ ]:
all_scan_data.img_dict

In [ ]:
plt.imshow(cropped_images[4])
# Skeletonize the grains
SKELETONIZE_METHOD = "zhang"
skeletons = [morphology.skeletonize(grain, method=SKELETONIZE_METHOD) for grain in cropped_masks]

In [ ]:
from skimage import feature, filters

In [ ]:
plt.imshow(cropped_masks[4])

In [ ]:
plt.imshow(skeletons[4])

## Gaussian on Mask First


Consider applying Gaussian on the mask first and then using an arbitrary threshold to dichotomise before then
skeletonising, this may give a cleaner mask to skeletonise, for example the below already looks cleaner.

In [ ]:
smoothed = filters.gaussian(cropped_images[4], sigma=3)
plt.imshow(smoothed)

# skan - Mask based Skeleton

Now we use [skan](https://github.com/jni/skan) to prune the skeleton that has been derived. This is done on the mask of
the skeleton .

In [ ]:
# Save the skeleton to PNG for use elsewhere (looks weird as its all black but it is there)
import imageio

imageio.imwrite("example_skeleton.png", skeletons[4].astype(np.uint8))

In [ ]:
import skan

# Create an object of type skan.Skeleton, this has the paths of each edge characterised.
skeleton = skan.Skeleton(skeletons[4])
# Extract all path (edge) co-ordinates
all_paths = [skeleton.path_coordinates(i) for i in range(skeleton.n_paths)]
# Summarize the skeleton
paths_table = skan.summarize(skeleton)
# Generate a random colour for each segment so they are distinguishable in Napari
paths_table["random-path-id"] = np.random.default_rng().permutation(skeleton.n_paths)

## Plotting with Napari

[napari](https://github.com/napari/napari) is a fast, interactive, multi-dimensional image viewer for Python (see also [Napari](https://napari.org)). 

In [ ]:
import napari

viewer = napari.Viewer(ndisplay=2)

skeleton_layer = viewer.add_shapes(
    all_paths,
    shape_type="path",
    properties=paths_table,
    edge_width=0.5,
    edge_color="random-path-id",
    edge_colormap="tab10",
)

## Prepare Skeleton for Pruning

In [ ]:
def prep_plot(skeleton):
    """Function for preparing to plot a skan skeleton with napari.

    Parameters
    ----------

    skeleton: Skeleton
        skan.Skeleton which has edges and paths defined.

    Returns
    -------
    tuple: all_paths, paths_table
        all_paths is the co-ordinates for all paths/edges in the plot, paths_table is a summary of the skeleton,
        augmented with random colours for each edge.
    """
    paths_table = skan.summarize(skeleton)
    all_paths = [skeleton.path_coordinates(i) for i in range(skeleton.n_paths)]
    paths_table["random-path-id"] = np.random.default_rng().permutation(skeleton.n_paths)
    return all_paths, paths_table

In [ ]:
def iteratively_prune_paths(
    skeleton: Skeleton, remove_branch_type: int = 1, spacing: float = 1e-9, value_is_height=False
) -> Skeleton:
    """Iteratively prune a skeleton using recursion removing the specified branch type.

    Will repeatedly remove branches of the specified type until there are none left on the Skeleton
    Parameters
    ----------
    remove_branch_type: int
        Remove branches of the specified type options are the types returned under `branch-type` by summarize and
    the default is `1` which removes junction-to-endpoint branches.

          0 endpoint-to-endpoint (isolated branch)
          1 junction-to-endpoint
          2 juntciont-to-junction
          3 isolated cycle

    Returns
    -------
    Skeleton
        Returns a new Skeleton instance.
    """
    pruned = Skeleton(skeleton, spacing=spacing, value_is_height=value_is_height)
    branch_data = summarize(pruned)
    iteration = 0
    while remove_branch_type in branch_data["branch-type"].unique():
        # pruned = pruned.prune_paths(branch_data.loc[branch_data["branch-type"] == remove_branch_type].index)
        # Remove branches that are small loops
        print(branch_data.loc[branch_data["branch-type"] == 3])
        print(f"BEFORE 1 : {pruned}")
        pruned = pruned.prune_paths(branch_data.loc[branch_data["branch-type"] == 3].index)
        print(f"BEFORE 2 : {pruned}")
        pruned = pruned.prune_paths(branch_data.loc[branch_data["branch-type"] == 1].index)
        print(f"BEFORE 3 : {pruned}")

        branch_data = summarize(pruned)
        iteration += 1
    return pruned
    # return skeleton

In [ ]:
print(f"paths_table.shape : {paths_table.shape}")
indexes = np.array(list(paths_table[paths_table["branch-type"] == 3].index))
print(f"Original indexes : {indexes}")
# print(f"index + 2        : {indexes + 2}")
indexes + 2

In [ ]:
skeleton = skan.Skeleton(skeletons[4])
all_paths, paths_table = prep_plot(skeleton)
viewer = napari.Viewer(ndisplay=3)
skeleton_layer = viewer.add_shapes(
    all_paths,
    shape_type="path",
    properties=paths_table,
    edge_width=0.5,
    edge_color="random-path-id",
    edge_colormap="tab10",
)

In [ ]:
skeleton.paths.indices

In [ ]:
import skan
from skan import draw
from skan.csr import skeleton_to_csgraph
from skan import Skeleton, summarize

pruned_skeleton = skeleton.iteratively_prune_paths(remove_branch_type=3)
# pruned_skeleton = iteratively_prune_paths(skeletons[4])
all_paths, paths_table = prep_plot(pruned_skeleton)
# viewer = napari.Viewer(ndisplay=2)
# skeleton_layer = viewer.add_shapes(
#         all_paths,
#         shape_type='path',
#         properties=paths_table,
#         edge_width=0.5,
#         edge_color='random-path-id',
#         edge_colormap='tab10',
# )
plt.imshow(pruned_skeleton)

# Ridge Based Approach

We now use Gaussian blurring to average the image and remove some of the noise before then determining the shape of the
region so that we can detect the ridge

**THOUGHT** The image below showsome regularish patterns on the edges this is presumably related to the helical nature
of the DNA. The turning circle of the DNA double helix is fairly well characterised I believe (CITATION NEEDED), could
this be used to help infer/ellucidate the angle/twisting/features of the molecul?e

In [ ]:
smoothed = filters.gaussian(cropped_images[4], sigma=3)
shape_index = feature.shape_index(smoothed)
plt.imshow(shape_index)

## Plotting Ridge Height

In [ ]:
shape_index * 

In [ ]:
# Multiply binary skeleton by the shape_index of the image
skeleton_height = skan.Skeleton(skeletons[4] * shape_index)
paths_shape, table_shape = prep_plot(skeleton_height)
table_shape["ridgeness"] = np.abs(0.5 - table_shape["mean-pixel-value"])
plt.imshow(skeleton_height)
plt.imshow(skeletons[4] * shape_index)

In [ ]:
viewer = napari.Viewer(ndisplay=2)
viewer.add_shapes(
    paths_shape,
    shape_type="path",
    properties=table_shape,
    edge_width=0.5,
    edge_color="ridgeness",
    edge_colormap="viridis",
)

In [ ]:
pruned_skeleton_height = skeleton_height.iteratively_prune_paths(remove_branch_type=1)
paths_shape, table_shape = prep_plot(pruned_skeleton_height)
viewer = napari.Viewer(ndisplay=2)
viewer.add_shapes(
    paths_shape,
    shape_type="path",
    properties=table_shape,
    edge_width=0.5,
    edge_color="ridgeness",
    edge_colormap="viridis",
)
plt.imshow(pruned_skeleton_height)

In [ ]:
%debug

In [ ]:
viewer = napari.Viewer(ndisplay=3)
viewer.add_shapes(
    all_paths,
    shape_type="path",
    properties=paths_table,
    edge_width=0.5,
    edge_color="random-path-id",
    edge_colormap="tab10",
)

In [ ]:
# skeletons[4]
paths_table["mean-pixel-value"]

In [ ]:
paths_table["euclidean-distance"].unique()
paths_table[paths_table["euclidean-distance"] == 0]

## Gaussian on Mask First

Consider applying Gaussian on the mask first and then using an arbitrary threshold to dichotomise before then
skeletonising, this may give a cleaner mask to skeletonise, for example the below already looks cleaner.

In [ ]:
smoothed = [filters.gaussian(img, sigma=3) for img in cropped_images]
# smoothed = filters.gaussian(cropped_images[4], sigma=3)
plt.imshow(smoothed[4])

In [ ]:
# OTSU threshold
threshold = [img >= filters.threshold_otsu(img) for img in smoothed]
# threshold = filters.threshold_otsu(smoothed)
# smoothed_mask = smoothed >= threshold
# plt.imshow(smoothed_mask)
plt.imshow(threshold[4])

In [ ]:
smoothed_skeleton = morphology.skeletonize(smoothed_mask, method=SKELETONIZE_METHOD)
plt.imshow(smoothed_skeleton)

In [ ]:
def iteratively_prune_paths(
    skeleton: Skeleton, remove_branch_type: int = 1, spacing: float = 1e-9, value_is_height=False
) -> Skeleton:
    """Iteratively prune a skeleton using recursion removing the specified branch type.

    Will repeatedly remove branches of the specified type until there are none left on the Skeleton
    Parameters
    ----------
    remove_branch_type: int
        Remove branches of the specified type options are the types returned under `branch-type` by summarize and
    the default is `1` which removes junction-to-endpoint branches.

          0 endpoint-to-endpoint (isolated branch)
          1 junction-to-endpoint
          2 juntciont-to-junction
          3 isolated cycle

    Returns
    -------
    Skeleton
        Returns a new Skeleton instance.
    """
    pruned = Skeleton(skeleton, spacing=spacing, value_is_height=value_is_height)
    branch_data = summarize(pruned)
    iteration = 0
    while remove_branch_type in branch_data["branch-type"].unique():
        # pruned = pruned.prune_paths(branch_data.loc[branch_data["branch-type"] == remove_branch_type].index)
        # Remove branches that are small loops
        print(branch_data.loc[branch_data["branch-type"] == 3])
        print(f"BEFORE 1 : {len(pruned.paths.indices)}")
        pruned = pruned.prune_paths(branch_data.loc[branch_data["branch-type"] == 1].index)
        pruned = Skeleton(pruned.skeleton_image, spacing=spacing, value_is_height=value_is_height)
        branch_data = summarize(pruned)
        print(f"BEFORE 2 : {len(pruned.paths.indices)}")
        pruned = pruned.prune_paths(branch_data.loc[branch_data["branch-type"] == 3].index)
        pruned = Skeleton(pruned.skeleton_image, spacing=spacing, value_is_height=value_is_height)
        branch_data = summarize(pruned)
        iteration += 1
    return pruned
    # return skeleton

In [ ]:
skel = Skeleton(smoothed_skeleton)
summary = summarize(skel)
skel.prune_paths(indices=summary.loc[summary["branch-type"] == 1])

In [ ]:
smoothed_skeleton_pruned = iteratively_prune_paths(smoothed_skeleton)
plt.imshow(smoothed_skeleton_pruned)